# Data as numbers: features, labels and tensors
**Machine Learning · Week 1 (Learning from examples) · Day 2**

Lesson page: https://sara-aminpour.com/tutorials/ml/w1d2.html

A dataset is a table of numbers. Today we put one into a PyTorch **tensor**,
slice it, summarize it, and look at it.

In [ ]:
import torch
from sklearn.datasets import load_iris

iris = load_iris()
X = torch.tensor(iris.data, dtype=torch.float32)   # features: one row per flower
y = torch.tensor(iris.target)                      # labels: 0, 1 or 2

print("X:", tuple(X.shape), X.dtype)
print("y:", tuple(y.shape), y.dtype)
print(X[:3])

## Rows, columns and masks
`X[i]` is a row, `X[:, j]` is a column, and `X[y == 0]` keeps the rows where the test is true.

In [ ]:
first = X[0]                 # one example: a vector of 4 features
petal_length = X[:, 2]       # one feature, for every example
setosa = X[y == 0]           # a boolean mask: the rows whose label is 0

print("one example: ", first, tuple(first.shape))
print("one feature: ", petal_length[:5], "...", tuple(petal_length.shape))
print("all setosa:  ", tuple(setosa.shape))

## Summaries and standardizing
Each feature has its own scale. Standardizing gives every column mean 0 and spread 1:
$z = (x - \mu)/\sigma$. PyTorch subtracts the row of means from **every** row at once
(this is called *broadcasting*).

In [ ]:
mean = X.mean(dim=0)         # dim=0: average down the rows, one number per column
std = X.std(dim=0)
Z = (X - mean) / std

print("means:          ", mean.round(decimals=2))
print("spreads (std):  ", std.round(decimals=2))
print("after: means    ", Z.mean(dim=0).round(decimals=2).abs())
print("after: spreads  ", Z.std(dim=0).round(decimals=2))

## Categories become one-hot vectors

In [ ]:
import torch.nn.functional as F

onehot = F.one_hot(y, num_classes=3)
for i in [0, 50, 100]:
    print(f"flower {i:3d}: label {y[i].item()} = {iris.target_names[y[i]]:<10}  one-hot {onehot[i].tolist()}")

## Every pair of features
The same picture as the widget, for all six pairs at once.

In [ ]:
import matplotlib.pyplot as plt

names = ["sepal length", "sepal width", "petal length", "petal width"]
colors = ["#ff8a50", "#5577f5", "#1fb394"]
fig, axes = plt.subplots(4, 4, figsize=(8, 8), layout="constrained")
for r in range(4):
    for c in range(4):
        ax = axes[r, c]
        if r == c:
            for k in range(3):
                ax.hist(X[y == k, c].numpy(), bins=12, alpha=0.6, color=colors[k])
        else:
            for k in range(3):
                ax.scatter(X[y == k, c], X[y == k, r], s=8, color=colors[k], label=iris.target_names[k])
        if r == 3:
            ax.set_xlabel(names[c])
        if c == 0:
            ax.set_ylabel(names[r])
        ax.tick_params(labelsize=7)
axes[0, 1].legend(fontsize=7, loc="upper right")
plt.show()

## Your turn: the average flower
Compute the mean of each of the 4 features over the rows of **one species at a time**.
Which feature differs the most between the species?

In [ ]:
for k, name in enumerate(iris.target_names):
    rows = X[y == k]
    average = None   # TODO: the mean of each column of `rows` (4 numbers)
    print(name, average)

---
## Solution
Try the exercise yourself first: it's fine to get stuck for a minute. Then compare with this one (there are many good answers).

In [ ]:
for k, name in enumerate(iris.target_names):
    rows = X[y == k]
    average = rows.mean(dim=0)
    print(f"{name:>10}: sepal {average[0]:.2f} x {average[1]:.2f} cm, petal {average[2]:.2f} x {average[3]:.2f} cm")

print("\nPetal length changes the most (1.46 -> 4.26 -> 5.55 cm), which is why the petal plot separates the species so well.")